In [1]:
import pandas as pd
import sqlite3

## Для этого упражнения нужно:

1. Создай соединение с базой данных с помощью библиотеки sqlite3.
2. Получи схему таблицы test.
3. Получи первые десять строк таблицы test, чтобы посмотреть её структуру.
4. Найди минимальное значение дельты между первым коммитом и дедлайном соответствующей лабораторной для всех пользователей одним запросом.
    - Выполни объединение с таблицей deadlines.
    - Разницу отобрази в часах.
    - Не учитывай лабораторную project1 (у неё более длинные дедлайны - это выброс).
    - Сохрани значение в DataFrame df_min вместе с соответствующим uid.
5. Аналогично найди максимум (тоже одним запросом). Имя DataFrame - df_max.
6. Аналогично найди среднее (тоже одним запросом). На этот раз в DataFrame не должно быть колонки uid. Имя DataFrame - df_avg.
7. Протестируй гипотезу, что у пользователей, которые посещали ленту новостей несколько раз, дельта между первым коммитом и дедлайном ниже. Для этого посчитай коэффициент корреляции между числом просмотров ленты и дельтой.
    - Одним запросом создай таблицу с колонками: "uid", "avg_diff", "pageviews".
    - "uid" - идентификаторы, существующие в test.
    - "avg_diff" - средняя дельта между первым коммитом и дедлайном по пользователю.
    - "pageviews" - число посещений ленты новостей по пользователю.
    - Не учитывай project1.
    - Сохрани результат в DataFrame views_diff.
    - Используй метод Pandas corr() для вычисления коэффициента корреляции между числом просмотров и дельтой.
8. Закрой соединение.

In [2]:
db_path = '../data/checking-logs.sqlite'
conn = sqlite3.connect(db_path)

In [3]:
schema = pd.io.sql.read_sql('PRAGMA table_info(test);', conn)
print('===Схема таблицы test===')
schema

===Схема таблицы test===


,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


In [4]:
first_ten_obj = pd.io.sql.read_sql('SELECT * FROM test LIMIT 10;', conn)
print('===Первые 10 объектов===')
first_ten_obj

===Первые 10 объектов===


,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


In [5]:
print('===Схема таблицы deadlines===')
pd.io.sql.read_sql('PRAGMA table_info(deadlines);', conn)

===Схема таблицы deadlines===


,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,labs,TEXT,0,None,0
2,2,deadlines,INTEGER,0,None,0


In [6]:
df_min = pd.io.sql.read_sql(
    """
    SELECT t.uid,
           MIN(CAST((UNIXEPOCH(t.first_commit_ts) - d.deadlines) / 3600 AS INTEGER)) AS delta
    FROM test t
    JOIN deadlines d ON t.labname = d.labs
    WHERE t.labname != 'project1'
    GROUP BY t.uid
    ORDER BY delta
    LIMIT 1;
    """,
    conn
)
print('===Датафрейм df_min===')
df_min

===Датафрейм df_min===


,uid,delta
0,user_30,-202


In [7]:
df_max = pd.io.sql.read_sql(
    """
    SELECT t.uid,
           MAX(CAST((UNIXEPOCH(t.first_commit_ts) - d.deadlines) / 3600 AS INTEGER)) AS delta
    FROM test t
    JOIN deadlines d ON t.labname = d.labs
    WHERE t.labname != 'project1'
    GROUP BY t.uid
    ORDER BY delta DESC
    LIMIT 1;
    """,
    conn
)
print('===Датафрейм df_max===')
df_max

===Датафрейм df_max===


,uid,delta
0,user_25,-2


In [8]:
df_avg = pd.io.sql.read_sql(
    """
    SELECT AVG((UNIXEPOCH(first_commit_ts) - deadlines) / 3600) AS avg_diff
    FROM (
        SELECT DISTINCT t.uid, t.labname, t.first_commit_ts, d.deadlines
        FROM test t
        JOIN deadlines d ON t.labname = d.labs
        WHERE t.labname != 'project1'
    ) AS sub;
    """,
    conn
)

print('===Датафрейм df_avg===')
df_avg

===Датафрейм df_avg===


,avg_diff
0,-89.125


In [9]:
views_diff = pd.io.sql.read_sql(
    """
    SELECT sub.uid,
           sub.avg_diff,
           (SELECT COUNT(*) FROM pageviews p WHERE p.uid = sub.uid) AS pageviews
    FROM (
        SELECT t.uid,
               CAST((UNIXEPOCH(MIN(t.first_commit_ts)) - MIN(d.deadlines)) / 3600 AS INTEGER) AS avg_diff
        FROM test t
        JOIN deadlines d ON t.labname = d.labs
        WHERE t.labname != 'project1'
        GROUP BY t.uid
    ) AS sub;
    """,
    conn
)

print('===Датафрейм views_diff===')
views_diff

===Датафрейм views_diff===


,uid,avg_diff,pageviews
0,user_1,-6,28
1,user_10,-39,89
2,user_14,-200,143
3,user_17,-81,47
4,user_18,-4,3
5,user_19,-148,16
6,user_21,-126,10
7,user_25,-148,179
8,user_28,-98,149
9,user_3,-75,317


In [10]:
correlation = views_diff.corr(numeric_only=True, method='pearson')
print('===Корреляционная матрица (Пирсон)===')
print(correlation)

===Корреляционная матрица (Пирсон)===
           avg_diff  pageviews
avg_diff   1.000000  -0.062967
pageviews -0.062967   1.000000


In [11]:
conn.close()